### Qwen model check on Colab

In [ ]:
import os

REPO = "unbiasederudite/qwen2.5-coder-text2sql-qlora"  # GitHub repository
REF = "main"  # branch or tag
REPO_DIR = REPO.split("/")[-1]

os.environ.update(REPO=REPO, REF=REF, REPO_DIR=REPO_DIR)

In [ ]:
%%bash
set -eu
exec 2>&1
curl -fsSL https://raw.githubusercontent.com/$REPO/$REF/scripts/colab_setup.sh -o /tmp/colab_setup.sh
bash /tmp/colab_setup.sh
bash "/content/$REPO_DIR/scripts/download_data.sh" spider

In [ ]:
%cd -q /content/{REPO_DIR}

from huggingface_hub import snapshot_download

from text2sql.model import MODELS, generate, load_model

for name in MODELS:
    snapshot_download(name)

Show one dev sample (system, user and the gold assistant message) and the reply each model writes, with the GPU memory the model takes.

In [ ]:
import gc
from pathlib import Path
from textwrap import indent

import torch

from text2sql.data import build_samples
from text2sql.prompt import build_completion, build_prompt

sample = build_samples(Path("data/spider_data"), "dev")[0]
messages = build_prompt(sample)

for message in messages + build_completion(sample):
    print(f"{message['role']}:")
    print(indent(message["content"], "    "), end="\n\n")

for name in MODELS:
    model, tokenizer = load_model(name)
    reply = generate(model, tokenizer, messages)
    print(f"{name} ({torch.cuda.memory_allocated() / 1e9:.1f} GB):")
    print(indent(reply, "    "), end="\n\n")

    del model
    gc.collect()
    torch.cuda.empty_cache()